# 回帰アルゴリズム：不動産価格の予測

・不動産会社  
・数百の不動産を保有し、賃貸物件を顧客に貸し出す。  
・要望：`物件の賃貸価格決定を自動化したい`  
　→ 要望の背景を考えることが大事。以下、調査済みの背景情報  

【背景】
現在、物件の賃貸価格を業界経験豊富なメンバーが地域の相場などを調査した後、決定している。このプロセスは安定しているが、新しい物件が加速度的に増えていること、賃貸価格を決定できるメンバーが限られていることから、作業が追い付かなくなっている。そこで、AIでの自動化を検討したい。  

## フェーズA：ビジネス課題分析

### 解決したい課題は何か。

・要望：物件の賃貸価格決定を自動化したい  
　→ `物件の賃貸価格決定プロセスの効率化`と言い換えられる。 

### 予測したい値は何か。どんなアクションに使えるか。

「不動産の賃貸価格決定の自動化」→「物件の賃貸価格予測」  
予測したい値 = `物件の賃貸価格`  

この値を用いたアクション...  
業界経験豊富なメンバーがこの結果を見て最終確認をし、実際の家賃に反映される。  
「純粋な予測結果」だけでなく、「どんな特徴量からこの結果が算出されたか」「どのくらいの精度か」も大事。

### 特徴量として何が使えるか

会社のデータベースに保存されている『今までに管理していた物件情報とその賃貸価格』

## フェーズB：データ分析、機械学習

### データ収集・観察

In [1]:
import pandas as pd

df = pd.read_csv("realestate_train.csv")
df.head()

,rent_price,house_area,year_from_built,distance,built_date,balcony_area,house_structure,floor,total_floor
0,81000.0,22.627647,21,360.0,2001-04-01,0.0,RC,4.0,10.0
1,119000.0,29.487423,11,720.0,2011-03-01,0.0,RC,3.0,10.0
2,65000.0,13.960667,32,640.0,1990-03-01,0.0,RC,3.0,6.0
3,230000.0,79.860208,13,480.0,2009-03-01,0.0,RC,15.0,29.0
4,102000.0,34.471313,23,320.0,1999-05-01,0.0,RC,2.0,7.0


### 特徴量エンジニアリング

特徴量エンジニアリングとは、特徴量を生成すること。特徴量には、「元データをそのまま使う特徴量」と「元データを加工して生成する特徴量」の2種類がある。  
「加工して生成する特徴量」としては`①物件の過去１年間の賃貸価格の変動`や`②建物構造`がある。②はカテゴリデータになっており、モデルに入力することが出来ないため、加工する必要がある。  

今回は「元データをそのまま使う特徴量」のみ使う。

### アルゴリズム・評価指標の選定

今回のような『実数値の予測』では、`回帰アルゴリズム`を用いる。他の事例では、店舗の売上予測や需要予測など、多岐にわたる。  

そのなかでもどのアルゴリズムを使えばよいかというのは、１つの正解はないが、最初は`Ridge Regression`を使ってみることを推奨する。なぜなら、アルゴリズムや結果の解釈がしやすく、結果も安定することが多いから。

### モデルの学習

In [2]:
# 1. Ridge Regressionモデルのクラスを読み込む
from sklearn.linear_model import Ridge

# 2. 特徴量と正解データの設定
## 予測したい列(正解データ)
target_col = ["rent_price"]
## 使いたい特徴量
feature_cols = ["house_area", "distance"]

## モデル学習のために「X:特徴量」「y:正解データ」に分割する
y = df[target_col]
X = df[feature_cols]

# 3. モデル学習
## モデルの初期化 
model = Ridge()
## 学習 
model.fit(X, y)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


・Ridge(alpha=1.4)
　Ridgeクラスではこのようにハイパーパラメータを設定できる。ハイパーパラメータは「どのように学習を行うか」を指定するためのもの。

### モデルの評価

モデルの評価とは、このモデルで予測した結果がどのくらい当たっているのかを計測すること。一般的に、「学習したデータに対して予測を行い、正解と比較する」方法で評価を行う。

In [3]:
# 学習したモデルで、予測値を出す
df["pred_rent_price"] = model.predict(X)
# 予測値と正解値の表を作って比較する
df[["rent_price", "pred_rent_price"]].head()

,rent_price,pred_rent_price
0,81000.0,93257.478559
1,119000.0,102825.371568
2,65000.0,74754.139652
3,230000.0,198256.385738
4,102000.0,115551.833713


正解と予測を１つ１つ見るのは途方もないので、一般的には、`評価指標`を用いて計算する。  

今回は評価指標として、MAE(平均絶対値誤差)といった正解値と予測値の差の絶対値で平均を取った指標を使う。

In [4]:
from sklearn.metrics import mean_absolute_error

print(mean_absolute_error(df["rent_price"], df["pred_rent_price"]))

17763.514918603825


MAE=17763(正解値と予測値の差が平均して17763円)という値が「良いのか、悪いのか」を判断する基準はない。「最終的にこの予測結果がどう使われるか」によって変わってくる。  

今回のケースでは、`「新しい物件の賃貸の価格決定を行いたい」`という目的...  
物件の賃貸価格としては、17763円のズレは大きな差に思われる。  
ここで、「1.制度を許容範囲まで高める」「2.人間のサポートを前提としてプロセスを構築する」という２つの選択肢が出てくる。2はモデルの予測値はあくまで参考値で、そこから人間が微調整をするという方法。実際のプロジェクトでは1を時間や労力をかけて行うより、2を選ぶ方が現実的なことも多々ある。

### モデルを用いた予測

In [5]:
df_pred = pd.read_csv("realestate_pred.csv")
df_pred.head()

# rent_price列がないデータ

,house_area,year_from_built,distance,built_date,balcony_area,house_structure,floor,total_floor
0,27.688916,17,360.0,2005-04-01,0.0,RC,11.0,11.0
1,34.501054,1,1040.0,2021-08-01,0.0,RC,4.0,7.0
2,20.243089,34,1000.0,1988-03-01,0.0,鉄骨,1.0,4.0
3,50.870947,33,520.0,1989-06-01,0.0,RC,4.0,4.0
4,65.707831,30,480.0,1992-01-01,0.0,鉄骨,3.0,4.0


In [8]:
# モデル学習時と同じ順番で同じ特徴量を渡さなければならない。
feature_cols = ["house_area", "distance"]

X_pred = df_pred[feature_cols]
X_pred["pred_rent_price"] = model.predict(X_pred)

X_pred.head()

,house_area,distance,pred_rent_price
0,27.688916,360.0,102635.524754
1,34.501054,1040.0,109321.714832
2,20.243089,1000.0,83252.251785
3,50.870947,520.0,144192.887622
4,65.707831,480.0,172033.388967
